# Assignment 1: Sampling and Reproducibility

The code at the end of this file explores contact tracing data about an outbreak of the flu, and demonstrates the dangers of incomplete and non-random samples. This assignment is modified from [Contact tracing can give a biased sample of COVID-19 cases](https://andrewwhitby.com/2020/11/24/contact-tracing-biased/) by Andrew Whitby.

Examine the code below. Identify all stages at which sampling is occurring in the model. Describe in words the sampling procedure, referencing the functions used, sample size, sampling frame, any underlying distributions involved. 



### Sampling stages
- Sample frame and size: 1,000 attendees (200 wedding, 800 brunch) built in `events` list; observational units are individuals.
- Sample procedure: `np.random.choice` selects 10% without replacement (attack rate) from the full frame, approximating a simple random sample of 100 infected people.
- Sampling function and distribution: Primary tracing sample: among the infected subset, `np.random.rand(...) < TRACE_SUCCESS` draws a Bernoulli distribution sample (p = 0.20) of roughly 20 traced cases. Secondary tracing: events with at least 2 traced infections (`value_counts` + threshold) trigger a cluster-style census of all infected people from those events being marked traced. Proportions are then calculated by grouping infections/traced cases by event type, so the sampling units are individuals but event clusters drive the second-stage inclusion.


Modify the number of repetitions in the simulation to 10 and 100 (from the original 1000). Run the script multiple times and observe the outputted graphs. Comment on the reproducibility of the results.


### Repetitions at 10 and 100
 With 10 repetitions the two histograms jump around each run and a few draws can heavily sway the perceived wedding share because the effective sample of simulations is tiny. With 100 repetitions the shapes smooth out but still shift between runs, while the spread narrows compared with 10, yet outputs remain inconsistent because randomness is uncontrolled.
 Across both settings, rerunning the notebook yields different plots each time, showing the simulation is not reproducible as-is.


Alter the code so that it is reproducible. Describe the changes you made to the code and how they affected the reproducibility of the script. The script needs to produce the same output when run multiple times.


### Reproducibility 
 I seeded the random number generator (`RNG_SEED = 42`) and routed all random draws through `numpy.random.default_rng` so every run follows the same sequence. And then, I parameterized the number of simulation runs (`SIMULATION_RUNS = 100`) to make it easy to switch between 10 and 100 while keeping results fixed for a given seed. After these changes, rerunning the notebook produces identical infection/trace proportions and histograms, satisfying the reproducibility requirement.


## Code

In [ ]:

# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

plt.switch_backend("Agg")

warnings.simplefilter(action='ignore', category=FutureWarning)

# Constants representing the parameters of the model
ATTACK_RATE = 0.10
TRACE_SUCCESS = 0.20
SECONDARY_TRACE_THRESHOLD = 2
SIMULATION_RUNS = 100
RNG_SEED = 42

def simulate_event(rng):
    """
    Simulates the infection and tracing process for a series of events.

    This function creates a DataFrame representing individuals attending weddings and brunches,
    infects a subset of them based on the ATTACK_RATE, performs primary and secondary contact tracing,
    and calculates the proportions of infections and traced cases that are attributed to weddings.

    Parameters:
    - rng: numpy Generator used for reproducible random draws.

    Returns:
    - A tuple containing the proportion of infections and the proportion of traced cases
      that are attributed to weddings.
    """
    # Create DataFrame for people at events with initial infection and traced status
    events = ['wedding'] * 200 + ['brunch'] * 800
    ppl = pd.DataFrame({
        'event': events,
        'infected': False,
        'traced': np.nan  # Initially setting traced status as NaN
    })

    # Explicitly set 'traced' column to nullable boolean type
    ppl['traced'] = ppl['traced'].astype(pd.BooleanDtype())

    # Infect a random subset of people
    infected_indices = rng.choice(ppl.index, size=int(len(ppl) * ATTACK_RATE), replace=False)
    ppl.loc[infected_indices, 'infected'] = True

    # Primary contact tracing: randomly decide which infected people get traced
    ppl.loc[ppl['infected'], 'traced'] = rng.random(sum(ppl['infected'])) < TRACE_SUCCESS

    # Secondary contact tracing based on event attendance
    event_trace_counts = ppl[ppl['traced'] == True]['event'].value_counts()
    events_traced = event_trace_counts[event_trace_counts >= SECONDARY_TRACE_THRESHOLD].index
    ppl.loc[ppl['event'].isin(events_traced) & ppl['infected'], 'traced'] = True

    # Calculate proportions of infections and traces attributed to each event type
    ppl['event_type'] = ppl['event'].str[0]  # 'w' for wedding, 'b' for brunch
    wedding_infections = sum(ppl['infected'] & (ppl['event_type'] == 'w'))
    brunch_infections = sum(ppl['infected'] & (ppl['event_type'] == 'b'))
    p_wedding_infections = wedding_infections / (wedding_infections + brunch_infections)

    wedding_traces = sum(ppl['infected'] & ppl['traced'] & (ppl['event_type'] == 'w'))
    brunch_traces = sum(ppl['infected'] & ppl['traced'] & (ppl['event_type'] == 'b'))
    p_wedding_traces = wedding_traces / (wedding_traces + brunch_traces)

    return p_wedding_infections, p_wedding_traces


def run_simulations(num_runs=SIMULATION_RUNS, seed=RNG_SEED):
    """Run the simulation multiple times with a fixed seed for reproducibility."""
    rng = np.random.default_rng(seed)
    results = [simulate_event(rng) for _ in range(num_runs)]
    return pd.DataFrame(results, columns=["Infections", "Traces"])


# Run the simulation and plot results
props_df = run_simulations()

plt.figure(figsize=(10, 6))
sns.histplot(props_df['Infections'], color="blue", alpha=0.75, binwidth=0.05, kde=False, label='Infections from Weddings')
sns.histplot(props_df['Traces'], color="red", alpha=0.75, binwidth=0.05, kde=False, label='Traced to Weddings')
plt.xlabel("Proportion of cases")
plt.ylabel("Frequency")
plt.title("Impact of Contact Tracing on Perceived Flu Infection Sources")
plt.legend()
plt.tight_layout()
plt.show()


## Criteria

|Criteria|Complete|Incomplete|
|--------|----|----|
|Alteration of the code|The code changes made, made it reproducible.|The code is still not reproducible.|
|Description of changes|The author answered questions and explained the reasonings for the changes made well.|The author did not answer questions or explain the reasonings for the changes made well.|

## Submission Information
🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `23:59 - 06 January 2026`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This markdown file (`a1_sampling_and_reproducibility.ipynb`) should be populated with the code changed.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/sampling/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

#### Checklist:
- [ ] Create a branch called `assignment-1`.
- [ ] Ensure that the repository is public.
- [ ] Review [the PR description guidelines](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md#guidelines-for-pull-request-descriptions) and adhere to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via the help channel in Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
